# 00 - Setup and Data Acquisition

**Project: Critical Stations - centrality and resilience of Israel's public transport network**

Israel's national public transport is published by the Ministry of Transport as a **GTFS**
feed (General Transit Feed Specification): a bundle of plain-text CSV files describing every
agency, route, trip, stop and scheduled stop time in the country. That feed is a graph in
disguise. If we treat every **stop** as a node and connect two stops whenever some scheduled
trip travels directly from one to the other, we get a national mobility network of roughly
30,000 nodes and 52,000 edges.

This notebook series asks two questions about that network:

1. **Which stations are critical?** Which stops carry a disproportionate share of the
   country's connectivity, according to degree, weighted degree, PageRank, betweenness,
   articulation points and bridges?
2. **How resilient is the network to losing them?** If the top-ranked stops are removed -
   flooding, a security incident, a strike, construction - how fast does the network
   fragment compared with losing random stops?

This first notebook is the **entry point**. It does no graph theory at all. Its job is to
prove that the environment works, to bring the one large data file that is not tracked in
git onto disk, and to give the reader an honest inventory of the raw data everything else
is built on.

**Inputs**

- `israel-public-transportation/` - the GTFS feed shipped with the repository
  (`agency.txt`, `calendar.txt`, `fare_attributes.txt`, `fare_rules.txt`, `routes.txt`,
  `stops.txt`, `translations.txt`, `trips.txt`)
- `stop_times.txt` (816 MB) - **not** in git; downloaded on demand from Google Drive

**Outputs** (all under `outputs/nb/00_setup_and_data/`)

- `tables/gtfs_file_inventory.csv` - every feed file with size, row count and purpose
- `tables/route_type_distribution.csv` - routes per GTFS `route_type`
- `tables/agency_route_counts.csv` - routes per operator
- `tables/service_calendar_summary.csv` - service days per weekday
- `feed_manifest.json` - machine-readable summary consumed by later notebooks
- `figures/route_type_distribution.png`, `figures/routes_per_agency.png`,
  `figures/stop_locations.png`

**Depends on:** nothing. This is stage 00 - run it first.

**Runtime:** a few seconds, plus a one-time ~816 MB download if `stop_times.txt` is absent.

## 1. Environment bootstrap

The cell below is the only piece of boilerplate in the project and it is repeated verbatim
at the top of every notebook. It does three things:

- `_ensure(...)` installs a package **only if the import cannot be resolved**, so re-running
  a notebook on a machine that is already set up costs nothing and never touches the
  network.
- `find_repo_root()` walks up from the current directory looking for the
  `israel-public-transportation/` folder. That makes the notebook work whether you launch
  Jupyter from the repo root, from `notebooks/`, or from anywhere else. If it cannot find
  the folder - which is what happens on a fresh Google Colab runtime - it clones the public
  repository into `/content` instead. A grader therefore needs nothing but this notebook
  and an internet connection.
- It fixes the working directory and defines `DATA` (raw feed) and `OUT` (notebook output
  root), so every later cell can use plain relative-free `Path` objects.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Dependencies and version check

The whole project runs on a small, ordinary scientific-Python stack: **pandas** for the CSV
tables, **numpy** for the numerics, **matplotlib** for every figure, and **networkx** for
the graph work in later notebooks. We install them here (no-op if already present) and then
print the versions.

Printing versions is not decoration. It is the fastest way for a grader to see that the
environment really is live, and if a result ever fails to reproduce, the version banner is
the first thing worth comparing.

In [ ]:
_ensure("pandas", "numpy", "matplotlib", "networkx")

import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print("python     ", sys.version.split()[0])
print("pandas     ", pd.__version__)
print("numpy      ", np.__version__)
print("matplotlib ", matplotlib.__version__)
print("networkx   ", nx.__version__)
print("\nData dir exists:", DATA.is_dir(), "->", DATA)

## 3. Output convention and run-time switches

Every notebook in this series writes into **its own** stage folder under `outputs/nb/`, and
reads earlier stages from theirs. Nothing is written into `outputs/tables`,
`outputs/figures` or `outputs/rail` - those hold the results already cited in the written
report and must stay untouched.

Two constants control the only non-trivial cost in this notebook:

- `COUNT_ROWS` - counting the rows of `stop_times.txt` means streaming 816 MB off disk
  (roughly 5-30 seconds depending on the drive). Set it to `False` for an instant run; the
  inventory then reports the row count as `NaN` for that one file.
- `PREVIEW_ROWS` / `TOP_AGENCIES` - purely cosmetic display limits.

In [ ]:
STAGE = OUT / "00_setup_and_data"
(STAGE / "tables").mkdir(parents=True, exist_ok=True)
(STAGE / "figures").mkdir(parents=True, exist_ok=True)

# --- run-time switches -------------------------------------------------------
COUNT_ROWS = True      # False -> skip the ~816 MB scan of stop_times.txt
PREVIEW_ROWS = 5       # rows shown in each preview table
TOP_AGENCIES = 12      # bars in the "routes per operator" figure

print("Stage output folder:", STAGE)

## 4. Fetching the raw feed (`stop_times.txt`)

Eight of the nine GTFS files are small enough to live in the repository. The ninth,
`stop_times.txt`, is **816 MB** - one row for every single scheduled arrival at every stop
in the country, about 15.7 million rows - and git is the wrong place for that. It is hosted
on Google Drive and pulled on demand with `gdown`.

This file is the raw material of the graph: consecutive rows sharing a `trip_id` are exactly
the "vehicle went directly from stop A to stop B" events that become edges in notebook 02.
The download runs once; on any later run the `exists()` check short-circuits it.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## 5. Hebrew text in figures

Every stop name, route description and operator name in this feed is Hebrew, and Hebrew is
written right-to-left. Matplotlib renders a string in logical (storage) order and does not
apply the Unicode bidirectional algorithm, so a Hebrew label drawn naively comes out
letter-reversed and unreadable to anyone who can actually read it.

The fix is to reorder the characters into *display* order before they reach the renderer,
using `python-bidi`. Rather than remembering to wrap every label by hand, we monkey-patch
`matplotlib.text.Text.set_text` once so that any Hebrew string is converted automatically,
everywhere - tick labels, titles, annotations. The patch is idempotent (guarded by
`_bidi_patched`) and leaves non-Hebrew text completely untouched.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 6. What GTFS is, and how the pieces fit together

GTFS is a relational schema stored as CSV files. Understanding the joins is the whole trick,
because the graph we care about only emerges after two of them:

```
agency.txt   1 --- *  routes.txt   1 --- *  trips.txt   1 --- *  stop_times.txt  * --- 1  stops.txt
 (operator)             (a line)              (one run of         (one arrival at        (a physical
                                               that line on        one stop, with         stop / platform)
                                               a given service)    a sequence number)
calendar.txt 1 --- * trips.txt        (which days a service_id actually runs)
fare_rules / fare_attributes          (ticketing - price zones, not topology)
translations.txt                      (Hebrew <-> other-language name strings)
```

Read bottom-up: a **stop** is a physical location with coordinates. A **stop_time** says
"trip T reached stop S as its k-th call, at 07:14". A **trip** is one physical run of a
**route** on a given **service** (calendar) day, and a route belongs to an **agency**.

The edge-building rule used throughout this project falls straight out of that schema:

> Sort `stop_times.txt` by `trip_id`, then `stop_sequence`. Any two **consecutive rows of
> the same trip** describe a vehicle travelling directly from one stop to the next - that is
> a directed edge. The number of trips using the same ordered stop pair becomes the edge
> weight (`frequency`).

Because the published feed already arrives sorted that way, the file can be streamed row by
row and never has to be held in memory - which matters a great deal at 15.7 million rows.

**Which files this project actually uses:**

| File | Used? | Role in this project |
|---|---|---|
| `stop_times.txt` | **yes, core** | Source of every edge: consecutive stops within a trip |
| `stops.txt` | **yes, core** | Node attributes: `stop_id`, Hebrew name, lat/lon, zone |
| `trips.txt` | **yes** | Maps trips to routes and services; lets us slice by mode |
| `routes.txt` | **yes** | `route_type` (bus / rail / light rail) - drives the rail-only analysis |
| `agency.txt` | yes, context | Operator names for descriptive breakdowns |
| `calendar.txt` | yes, context | Confirms which service window the snapshot covers |
| `translations.txt` | no | Multi-language name strings; we work in Hebrew directly |
| `fare_rules.txt` | no | Ticketing zones - a pricing relation, not a topological one |
| `fare_attributes.txt` | no | Ticket prices - same reason |

Fares are deliberately out of scope. A fare zone tells you what a journey costs, not whether
the journey is physically possible, and this project is about connectivity and failure.

## 7. Inventory of all nine feed files

Now we prove the data is really there. For each of the nine files we record its size on
disk, its column list, and its data-row count.

Counting rows needs a little care. `pd.read_csv` on an 816 MB file would work but is wasteful
when all we want is a count, so instead we stream the file in 8 MB binary chunks and count
newline bytes, subtracting one for the header (and handling a missing final newline). One
caveat, stated honestly: this counts *lines*, not CSV records, so a field containing a
quoted embedded newline would be over-counted. Spot checks against the parsed tables show
this feed contains none.

The `purpose` column is our own annotation - the "used / not used" judgement from the schema
section above, attached to the data so later stages and the report can cite it.

In [ ]:
FILE_PURPOSE = {
    "stop_times.txt":      ("core",    "One row per scheduled arrival. Consecutive rows of a trip define graph edges."),
    "stops.txt":           ("core",    "Graph nodes: stop_id, Hebrew name, latitude/longitude, fare zone."),
    "trips.txt":           ("used",    "Links each trip to its route and service_id (calendar)."),
    "routes.txt":          ("used",    "Route metadata incl. route_type (bus=3, rail=2, light rail=0)."),
    "agency.txt":          ("context", "Operators (Israel Railways, Egged, Dan, ...)."),
    "calendar.txt":        ("context", "Weekly service pattern and validity window per service_id."),
    "translations.txt":    ("unused",  "Multi-language name strings; the project works in Hebrew directly."),
    "fare_rules.txt":      ("unused",  "Fare zone rules - pricing, not topology."),
    "fare_attributes.txt": ("unused",  "Ticket prices - pricing, not topology."),
}

def count_data_rows(path, chunk=8 << 20):
    """Stream the file and count newline bytes; returns rows excluding the header."""
    newlines, last = 0, b"\n"
    with open(path, "rb") as fh:
        while True:
            buf = fh.read(chunk)
            if not buf:
                break
            newlines += buf.count(b"\n")
            last = buf[-1:]
    total_lines = newlines + (0 if last == b"\n" else 1)
    return max(total_lines - 1, 0)

def header_columns(path):
    """Read only the first line. utf-8-sig strips the BOM this feed ships with."""
    with open(path, "r", encoding="utf-8-sig", newline="") as fh:
        return fh.readline().rstrip("\r\n").split(",")

rows = []
missing = []
for name, (tier, purpose) in FILE_PURPOSE.items():
    path = DATA / name
    if not path.exists():
        missing.append(name)
        continue
    size_mb = path.stat().st_size / 1024**2
    do_count = COUNT_ROWS or size_mb < 200
    cols = header_columns(path)
    rows.append({
        "file": name,
        "size_mb": round(size_mb, 2),
        "rows": count_data_rows(path) if do_count else np.nan,
        "columns": len(cols),
        "tier": tier,
        "column_names": ", ".join(cols),
        "purpose": purpose,
    })

if missing:
    raise FileNotFoundError(
        "Missing GTFS file(s): " + ", ".join(missing) + "\n"
        f"Expected inside {DATA}. If stop_times.txt is the missing one, re-run the "
        "gdown cell above; otherwise re-clone the repository."
    )

inventory = pd.DataFrame(rows).sort_values("size_mb", ascending=False).reset_index(drop=True)
inventory.to_csv(STAGE / "tables" / "gtfs_file_inventory.csv",
                 index=False, encoding="utf-8-sig")

print(f"{len(inventory)} GTFS files, "
      f"{inventory['size_mb'].sum():,.1f} MB total, "
      f"{inventory['rows'].sum():,.0f} data rows\n")
inventory[["file", "size_mb", "rows", "columns", "tier", "purpose"]]

## 8. What service period does this snapshot cover?

A GTFS feed is a *snapshot*, and every claim we later make about "the network" is really a
claim about the timetable that was valid during a particular window. `calendar.txt` states,
for each `service_id`, which weekdays it runs on and between which dates it is valid
(`start_date` / `end_date`, formatted `YYYYMMDD`).

So we take the minimum `start_date` and the maximum `end_date` across all services to get
the outer envelope of the snapshot, and we count how many services operate on each weekday.
The weekday profile is worth a glance for a very Israeli reason: Saturday is Shabbat and
Friday afternoon is a partial day, so a healthy feed should show a visible drop on Friday
and Saturday. If it does, the data is behaving as expected.

In [ ]:
calendar = pd.read_csv(DATA / "calendar.txt", dtype=str,
                       keep_default_na=False, encoding="utf-8-sig")

start = pd.to_datetime(calendar["start_date"], format="%Y%m%d")
end = pd.to_datetime(calendar["end_date"], format="%Y%m%d")

feed_start, feed_end = start.min(), end.max()
span_days = (feed_end - feed_start).days + 1

DAYS = ["sunday", "monday", "tuesday", "wednesday", "thursday", "friday", "saturday"]
day_counts = pd.DataFrame({
    "weekday": DAYS,
    "services_running": [(calendar[d] == "1").sum() for d in DAYS],
})
day_counts["share_of_services"] = (day_counts["services_running"] / len(calendar)).round(3)
day_counts.to_csv(STAGE / "tables" / "service_calendar_summary.csv",
                  index=False, encoding="utf-8-sig")

print(f"service_id entries : {len(calendar):,}")
print(f"feed valid from    : {feed_start.date()}")
print(f"feed valid until   : {feed_end.date()}")
print(f"span               : {span_days} days\n")
day_counts

## 9. A first look at the three tables that build the graph

Numbers in a table are abstract until you see the actual rows, so here are the first few
records of the three files the graph is made of. Two loading choices are deliberate and are
used consistently in every later notebook:

- `dtype=str` - GTFS identifiers are *opaque strings*. Letting pandas guess would turn
  `stop_id` `"007"` into the integer `7`, and joins against a file that kept it as text
  would then silently fail. Everything stays text until we explicitly convert.
- `keep_default_na=False` - an empty GTFS field means "not supplied", not "missing number".
  Keeping it as `""` avoids `NaN` leaking into string columns.

Watch for the shape of the data: `stops.txt` gives coordinates and a Hebrew name per node,
`routes.txt` gives the mode via `route_type`, and `trips.txt` is the many-to-one bridge from
individual vehicle runs back to routes.

In [ ]:
def load_gtfs(name, nrows=None):
    """Load a GTFS table as raw strings (see markdown above for why)."""
    path = DATA / name
    if not path.exists():
        raise FileNotFoundError(f"{path} missing - see the setup cells above.")
    return pd.read_csv(path, dtype=str, keep_default_na=False,
                       encoding="utf-8-sig", nrows=nrows)

stops = load_gtfs("stops.txt")
routes = load_gtfs("routes.txt")
agency = load_gtfs("agency.txt")

print("stops.txt  ", stops.shape)
display(stops.head(PREVIEW_ROWS))

print("routes.txt ", routes.shape)
display(routes.head(PREVIEW_ROWS))

trips_head = load_gtfs("trips.txt", nrows=PREVIEW_ROWS)
print("trips.txt  (preview only - full file has ~420k rows)")
display(trips_head)

print("stop_times.txt (preview only - full file has ~15.7M rows)")
display(pd.read_csv(STOP_TIMES, dtype=str, keep_default_na=False,
                    encoding="utf-8-sig", nrows=PREVIEW_ROWS))

## 10. Composition of the feed: modes and operators

Two descriptive breakdowns set expectations for everything that follows.

**By `route_type`** (the GTFS mode code: 0 = tram/light rail, 2 = heavy rail, 3 = bus, and
so on). This matters because the national network is overwhelmingly bus, which means the
combined graph's structure is essentially the bus structure. That is exactly why the project
also runs a separate rail-only analysis - heavy rail is a small, sparse, nearly-linear
subnetwork whose resilience profile is completely different, and it would otherwise be
drowned out.

**By operator.** Israel's bus network is franchised to many regional operators, so a route
count per agency shows how fragmented the system is. This is also the first figure with
Hebrew labels, so it doubles as a live test of the bidi patch from section 5 - if the
operator names read correctly right-to-left, the plotting stack is fully working.

In [ ]:
ROUTE_TYPE_LABELS = {
    "0": "tram/light rail", "1": "subway", "2": "rail", "3": "bus",
    "4": "ferry", "5": "cable tram", "6": "aerial lift", "7": "funicular",
    "8": "trolleybus", "715": "demand/other bus",
}

# --- routes per mode ---------------------------------------------------------
route_types = (routes["route_type"].value_counts()
               .rename_axis("route_type").reset_index(name="routes"))
route_types["route_type_label"] = route_types["route_type"].map(ROUTE_TYPE_LABELS).fillna("unknown")
route_types = route_types[["route_type", "route_type_label", "routes"]]
route_types["share"] = (route_types["routes"] / route_types["routes"].sum()).round(4)
route_types.to_csv(STAGE / "tables" / "route_type_distribution.csv",
                   index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(7, 4))
labels = route_types["route_type_label"] + " (" + route_types["route_type"] + ")"
ax.bar(labels, route_types["routes"], color="#0f766e")
ax.set_ylabel("routes")
ax.set_title("Routes by GTFS route_type")
ax.tick_params(axis="x", rotation=30)
for lbl in ax.get_xticklabels():
    lbl.set_ha("right")
fig.tight_layout()
fig.savefig(STAGE / "figures" / "route_type_distribution.png", dpi=150)
plt.show()

display(route_types)

# --- routes per operator -----------------------------------------------------
agency_routes = (routes.merge(agency[["agency_id", "agency_name"]], on="agency_id", how="left")
                 .assign(agency_name=lambda d: d["agency_name"].replace("", "unknown"))
                 .groupby("agency_name").size()
                 .sort_values(ascending=False)
                 .rename("routes").reset_index())
agency_routes.to_csv(STAGE / "tables" / "agency_route_counts.csv",
                     index=False, encoding="utf-8-sig")

top = agency_routes.head(TOP_AGENCIES).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top["agency_name"], top["routes"], color="#1d4ed8")
ax.set_xlabel("routes")
ax.set_title(f"Top {TOP_AGENCIES} operators by number of routes")
fig.tight_layout()
fig.savefig(STAGE / "figures" / "routes_per_agency.png", dpi=150)
plt.show()

print(f"{len(agency_routes)} operators in the feed")
display(agency_routes.head(TOP_AGENCIES))

## 11. Geographic sanity check

The last check is the most direct one: plot every stop by its longitude and latitude with no
map, no projection and no styling. If the coordinate columns are parsed correctly, the
outline of Israel should simply appear - the dense coastal strip from Ashkelon through Tel
Aviv to Haifa, the Jerusalem corridor branching inland, and the sparse Negev thinning out to
the south.

This is a genuinely useful validation and not just a pretty picture: swapped lat/lon columns,
a decimal-separator problem, or rows with placeholder `0,0` coordinates would all be
instantly visible here and invisible in a table of summary statistics. We count how many
stops have unusable coordinates rather than silently dropping them.

In [ ]:
coords = stops.copy()
coords["stop_lat"] = pd.to_numeric(coords["stop_lat"], errors="coerce")
coords["stop_lon"] = pd.to_numeric(coords["stop_lon"], errors="coerce")

bad = coords["stop_lat"].isna() | coords["stop_lon"].isna()
out_of_range = (~bad) & ~(coords["stop_lat"].between(29, 34) & coords["stop_lon"].between(33, 36))
print(f"stops total              : {len(coords):,}")
print(f"unparseable coordinates  : {int(bad.sum()):,}")
print(f"outside the Israel bbox  : {int(out_of_range.sum()):,}")

plot_df = coords[~bad & ~out_of_range]
fig, ax = plt.subplots(figsize=(6, 8))
ax.scatter(plot_df["stop_lon"], plot_df["stop_lat"], s=0.6, alpha=0.35,
           color="#0f766e", linewidths=0)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title(f"All {len(plot_df):,} GTFS stops")
ax.set_aspect(1.15)
fig.tight_layout()
fig.savefig(STAGE / "figures" / "stop_locations.png", dpi=150)
plt.show()

## 12. Writing the stage manifest

Finally we persist a small JSON manifest describing this feed snapshot: the file inventory,
the service window and the headline counts. Later notebooks load it to report *which*
snapshot their numbers came from, and it makes the whole series traceable to one dated
version of the data instead of "the GTFS feed" in the abstract.

Stage 00 deliberately produces no graph and no metric. Its only contract with the rest of
the series is: the feed is on disk, it parses, and here is what is in it.

In [ ]:
manifest = {
    "stage": "00_setup_and_data",
    "data_dir": str(DATA),
    "feed_service_start": str(feed_start.date()),
    "feed_service_end": str(feed_end.date()),
    "feed_span_days": int(span_days),
    "files": int(len(inventory)),
    "total_size_mb": round(float(inventory["size_mb"].sum()), 2),
    "stop_times_rows": (None if pd.isna(inventory.set_index("file").loc["stop_times.txt", "rows"])
                        else int(inventory.set_index("file").loc["stop_times.txt", "rows"])),
    "stops": int(len(stops)),
    "routes": int(len(routes)),
    "agencies": int(len(agency)),
    "service_ids": int(len(calendar)),
    "route_type_counts": dict(zip(route_types["route_type_label"], route_types["routes"].astype(int))),
}

with (STAGE / "feed_manifest.json").open("w", encoding="utf-8") as fh:
    json.dump(manifest, fh, ensure_ascii=False, indent=2)

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("\nArtifacts written:")
for p in sorted(STAGE.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(STAGE).as_posix())

## Takeaways

- **The environment works.** The dependency stack imports, the 816 MB `stop_times.txt` is on
  disk, all nine GTFS files parse, and Hebrew labels render in display order. Anything that
  fails later is an analysis problem, not a setup problem.
- **The feed is one snapshot, not "the network forever."** `calendar.txt` gives an explicit
  validity window, and every conclusion in the following notebooks is scoped to it. The
  weekday profile shows the expected Friday/Saturday drop, which is a good sign the data is
  intact.
- **The network is overwhelmingly bus.** Buses dominate the route count so heavily that the
  combined graph is, structurally, the bus graph. Heavy rail is a tiny fraction of the routes
  - which is precisely why it gets its own analysis rather than being read off the national
  numbers, where it is statistically invisible.
- **Only four of the nine files carry topology.** `stop_times.txt`, `stops.txt`, `trips.txt`
  and `routes.txt` build the graph; `agency.txt` and `calendar.txt` provide context; the two
  fare files and `translations.txt` are genuinely irrelevant to a connectivity question. It
  is worth saying plainly that a fare-weighted or time-weighted network would be a different
  and also interesting study - this project does not attempt it.
- **One honest limitation, stated up front:** the edges we are about to build encode *service
  topology*, not geography or travel time. An edge means "a scheduled vehicle goes directly
  from A to B", weighted by how many trips do so. It says nothing about how long that takes
  or how far it is. Every centrality and resilience result in this series must be read in
  those terms.

**Next:** notebook `02_graph_construction` streams `stop_times.txt` and turns those
consecutive-stop pairs into the weighted stop graph.